# AI-Driven Anomaly Detection for Securing Cloud-Native CI/CD Pipelines
#### **Created by: Nandana Bindhu **
This notebook implements an end-to-end anomaly detection pipeline that loads heterogeneous CSV datasets, preprocesses and aligns features, constructs temporal sequences, trains multiple anomaly detection models, evaluates results, and saves plots plus a Word report.

A CI/CD (Continuous Integration and Continuous Deployment) pipeline is a structured software delivery workflow that automates build, test, and deployment stages. In this context, anomaly detection is relevant for monitoring CI/CD logs, pipeline execution traces, and failure patterns to identify abnormal behavior early and support secure DevSecOps operations.


In [ ]:
from __future__ import annotations
import os

os.environ.setdefault("KERAS_BACKEND", "torch")

import matplotlib.pyplot as plt
from docx.shared import Inches
from docx import Document
from typing import List, Optional, Sequence, Tuple
from pathlib import Path
import warnings
import re
import math
import keras
import matplotlib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

matplotlib.use("Agg")

warnings.filterwarnings("ignore")


In [ ]:
if "__file__" not in globals():
    __file__ = str((Path.cwd() / "main.py").resolve())

SEED = 42
WINDOW_SIZE = 10
TEST_SIZE = 0.2
AUTOENCODER_EPOCHS = 8
VAE_EPOCHS = 10
BATCH_SIZE = 256
MAX_TRAIN_SAMPLES = 50000
MAX_FEATURE_COLUMNS = 64
MAX_ROWS_PER_FILE = 5000
OUTPUT_DIR = Path(__file__).resolve().parent / "outputs"
DATASET_DIRS = ["dataset1", "dataset2", "dataset3", "dataset4"]

LABEL_CANDIDATES = {
    "label",
    "class",
    "target",
    "anomaly",
    "anomaly_label",
    "is_anomaly",
    "incident_created",
    "failure",
    "outcome",
}

NORMAL_LABELS = {
    "0",
    "false",
    "benign",
    "normal",
    "clean",
    "safe",
    "ok",
    "none",
}

ANOMALY_LABELS = {
    "1",
    "true",
    "attack",
    "anomaly",
    "malicious",
    "failure",
    "incident",
    "fault",
    "error",
}




## Utility Functions: File Handling

These functions manage output paths, discover dataset CSV files, detect header format, standardize columns, and load/align individual files into a unified schema. They are needed to robustly handle heterogeneous datasets across folder trees.


In [ ]:
def setting_global_seed(seed: int = SEED) -> None:
    np.random.seed(seed)
    try:
        keras.utils.set_random_seed(seed)
    except Exception:
        pass


def ensure_output_directory() -> Path:
    OUTPUT_DIR.mkdir(exist_ok=True)
    return OUTPUT_DIR


def slugifying_model_name(model_name: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    return slug or "model"


def artifact_paths_for_model(model_name: str, output_dir: Path) -> dict:
    slug = slugifying_model_name(model_name)
    return {
        "confusion_matrix": output_dir / f"{slug}_cm.png",
        "roc_curve": output_dir / f"{slug}_roc.png",
        "loss_curve": output_dir / f"{slug}_loss.png",
        "anomaly_scores": output_dir / f"{slug}_scores.png",
    }


def discovering_csv_files(root_dirs: Sequence[str]) -> List[Path]:
    files: List[Path] = []
    for root_name in root_dirs:
        root = Path(__file__).resolve().parent / root_name
        if not root.exists():
            print(f"[WARN] Missing dataset folder: {root}")
            continue
        files.extend(sorted(root.rglob("*.csv")))
    return files


def finding_first_non_empty_line(path: Path) -> str:
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            stripped = line.strip()
            if stripped:
                return stripped
    return ""


def looking_for_header(line: str) -> bool:
    if not line:
        return True
    tokens = [token.strip().strip("\"'") for token in line.split(",")]
    if not tokens:
        return True
    alpha_tokens = sum(1 for token in tokens if re.search(
        r"[A-Za-z]", token or ""))
    if alpha_tokens >= max(2, int(0.2 * len(tokens))):
        return True
    header_markers = {"label", "class",
                      "timestamp", "time", "date", "id", "target"}
    token_words = {token.lower() for token in tokens}
    if token_words & header_markers:
        return True
    return False


def standardizing_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(column).strip() for column in df.columns]
    return df




In [ ]:
def infering_label_column(columns: Sequence[str]) -> Optional[str]:
    lowered = {str(column).strip().lower(): str(column) for column in columns}
    for candidate in ["label", "class", "target", "anomaly", "anomaly_label", "is_anomaly", "incident_created"]:
        if candidate in lowered:
            return lowered[candidate]
    for column in columns:
        lower = str(column).strip().lower()
        if lower in LABEL_CANDIDATES:
            return str(column)
        if "label" in lower or "class" in lower or lower.endswith("target"):
            return str(column)
    return None


def normalizing_label_series(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.strip().str.lower()
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().mean() >= 0.9:
        values = sorted(numeric.dropna().unique())
        if not values:
            return pd.Series([np.nan] * len(series), index=series.index, dtype="float64")
        if set(values).issubset({0, 1}):
            return numeric.fillna(0).astype(int)
        if len(values) == 2:
            normal_value = 0 if 0 in values else min(values)
            return (numeric != normal_value).astype(int)
        threshold = float(np.median(values))
        return (numeric > threshold).astype(int)

    mapped = cleaned.map(
        lambda value: 0
        if value in NORMAL_LABELS
        else 1
        if value in ANOMALY_LABELS
        else np.nan
    )
    if mapped.notna().mean() >= 0.5:
        return mapped.fillna(1).astype(int)

    # For multiclass labels, treat benign/normal as the only normal class.
    return cleaned.map(lambda value: 0 if value in NORMAL_LABELS else 1).astype(int)


def loading_single_csv_file(path: Path) -> Optional[pd.DataFrame]:
    try:
        header_present = looking_for_header(finding_first_non_empty_line(path))
        if header_present:
            frame = pd.read_csv(
                path,
                nrows=MAX_ROWS_PER_FILE,
                low_memory=False,
                on_bad_lines="skip",
                encoding_errors="ignore",
            )
        else:
            frame = pd.read_csv(
                path,
                header=None,
                nrows=MAX_ROWS_PER_FILE,
                low_memory=False,
                on_bad_lines="skip",
                encoding_errors="ignore",
            )
            frame.columns = [f"col_{index}" for index in range(frame.shape[1])]

        frame = standardizing_column_names(frame)
        if frame.empty:
            print(f"[WARN] Empty file skipped: {path}")
            return None

        label_column = infering_label_column(frame.columns)
        if label_column is not None:
            label_series = normalizing_label_series(frame[label_column])
            frame = frame.drop(columns=[label_column])
        elif not header_present and frame.shape[1] >= 1:
            label_series = normalizing_label_series(frame.iloc[:, -1])
            frame = frame.iloc[:, :-1].copy()
        else:
            label_series = pd.Series([np.nan] * len(frame), index=frame.index)

        feature_columns = [column for column in frame.columns if column not in {
            "label", "_source_file", "_source_folder", "_row_order"}]
        feature_columns = feature_columns[:MAX_FEATURE_COLUMNS]
        frame = frame[feature_columns].copy()

        standardized_feature_names = [
            f"feature_{index}" for index in range(MAX_FEATURE_COLUMNS)]
        rename_map = {column: standardized_feature_names[index]
                      for index, column in enumerate(feature_columns)}
        frame = frame.rename(columns=rename_map)
        for missing_index in range(len(feature_columns), MAX_FEATURE_COLUMNS):
            frame[standardized_feature_names[missing_index]] = np.nan

        frame = frame[standardized_feature_names]
        frame["label"] = label_series.reset_index(drop=True)

        frame["_source_file"] = path.name
        frame["_source_folder"] = path.parent.name
        frame["_row_order"] = np.arange(len(frame))
        return frame
    except Exception as exc:
        print(f"[WARN] Failed to load {path}: {exc}")
        return None


def loading_all_datasets(csv_files: Sequence[Path]) -> pd.DataFrame:
    frames: List[pd.DataFrame] = []
    failed = 0
    for index, path in enumerate(csv_files, start=1):
        print(f"[LOAD] {index}/{len(csv_files)} -> {path}")
        frame = loading_single_csv_file(path)
        if frame is None:
            failed += 1
            continue
        frames.append(frame)
    if not frames:
        raise RuntimeError("No CSV files could be loaded successfully.")
    combined = pd.concat(frames, ignore_index=True, sort=False)
    print(
        f"[LOAD] Loaded {len(frames)} files, skipped {failed} files, total rows: {len(combined)}")
    return combined


## Utility Functions: Label Processing

These functions infer where labels are stored and normalize them into binary anomaly targets. They are needed because source datasets use inconsistent naming and label conventions.


In [ ]:
def infering_label_column(columns: Sequence[str]) -> Optional[str]:
    lowered = {str(column).strip().lower(): str(column) for column in columns}
    for candidate in ["label", "class", "target", "anomaly", "anomaly_label", "is_anomaly", "incident_created"]:
        if candidate in lowered:
            return lowered[candidate]
    for column in columns:
        lower = str(column).strip().lower()
        if lower in LABEL_CANDIDATES:
            return str(column)
        if "label" in lower or "class" in lower or lower.endswith("target"):
            return str(column)
    return None


def normalizing_label_series(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.strip().str.lower()
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().mean() >= 0.9:
        values = sorted(numeric.dropna().unique())
        if not values:
            return pd.Series([np.nan] * len(series), index=series.index, dtype="float64")
        if set(values).issubset({0, 1}):
            return numeric.fillna(0).astype(int)
        if len(values) == 2:
            normal_value = 0 if 0 in values else min(values)
            return (numeric != normal_value).astype(int)
        threshold = float(np.median(values))
        return (numeric > threshold).astype(int)

    mapped = cleaned.map(
        lambda value: 0
        if value in NORMAL_LABELS
        else 1
        if value in ANOMALY_LABELS
        else np.nan
    )
    if mapped.notna().mean() >= 0.5:
        return mapped.fillna(1).astype(int)

    # For multiclass labels, treat benign/normal as the only normal class.
    return cleaned.map(lambda value: 0 if value in NORMAL_LABELS else 1).astype(int)




## Utility Functions: Preprocessing and Feature Engineering

These functions clean invalid values, remove non-informative identifiers, infer numeric vs categorical fields, and apply imputation/scaling/encoding. They are needed to produce model-ready numeric tensors from mixed raw CSV data.

From a CI/CD viewpoint, the raw data corresponds to pipeline logs and execution traces collected from build and deployment systems. The engineered feature space can represent operational indicators such as build status, execution duration, error frequency, stage-level outcomes, and other telemetry useful for anomaly detection in software delivery pipelines.


In [ ]:
def should_drop_column_from_data(column_name: str) -> bool:
    lower = str(column_name).strip().lower()
    patterns = [
        r"(^|_)id$",
        r"(^|_)uuid$",
        r"hash$",
        r"timestamp$",
        r"datetime$",
        r"^date$",
        r"^time$",
        r"_id$",
        r"commit_hash$",
        r"pipeline_id$",
        r"run_id$",
        r"error_message$",
    ]
    return any(re.search(pattern, lower) for pattern in patterns)


def align_and_cleaning_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    frame = dataframe.copy()
    frame.replace([np.inf, -np.inf], np.nan, inplace=True)
    frame.replace(r"^\s*$", np.nan, regex=True, inplace=True)
    feature_columns = [column for column in frame.columns if column not in {
        "label", "_source_file", "_source_folder", "_row_order"}]
    drop_columns = [
        column for column in feature_columns if should_drop_column_from_data(column)]
    if drop_columns:
        print(
            f"[PREP] Dropping non-useful columns: {drop_columns[:12]}{' ...' if len(drop_columns) > 12 else ''}")
        frame = frame.drop(columns=drop_columns, errors="ignore")

    return frame


def building_preprocessor(feature_frame: pd.DataFrame) -> Tuple[ColumnTransformer, List[str], List[str], pd.DataFrame]:
    numeric_columns: List[str] = []
    categorical_columns: List[str] = []

    working = feature_frame.copy()
    for column in working.columns:
        series = working[column]
        normalized = series.astype(str).str.strip().str.lower()
        boolean_like = normalized.isin(
            {"true", "false", "yes", "no", "0", "1"}).mean() >= 0.9
        if boolean_like:
            working[column] = normalized.map(
                {"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0}).astype(float)
            numeric_columns.append(column)
            continue

        converted = pd.to_numeric(series, errors="coerce")
        if converted.notna().mean() >= 0.6:
            working[column] = converted
            numeric_columns.append(column)
        else:
            working[column] = series.astype(str).replace(
                {"nan": np.nan, "none": np.nan, "": np.nan})
            categorical_columns.append(column)

    transformers = []
    if numeric_columns:
        transformers.append(
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_columns,
            )
        )
    if categorical_columns:
        transformers.append(
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value", unknown_value=-1),
                        ),
                    ]
                ),
                categorical_columns,
            )
        )

    if not transformers:
        raise RuntimeError(
            "No usable feature columns were found after cleaning.")

    preprocessor = ColumnTransformer(
        transformers=transformers, remainder="drop", sparse_threshold=0.0)
    return preprocessor, numeric_columns, categorical_columns, working


def preprocess_dataframe(dataframe: pd.DataFrame) -> Tuple[np.ndarray, Optional[np.ndarray], pd.DataFrame, ColumnTransformer]:
    cleaned = align_and_cleaning_features(dataframe)
    feature_frame = cleaned.drop(columns=["label"], errors="ignore")
    preprocessor, numeric_columns, categorical_columns, prepared_frame = building_preprocessor(
        feature_frame)
    print(
        f"[PREP] Numeric columns: {len(numeric_columns)}, categorical columns: {len(categorical_columns)}")
    transformed = preprocessor.fit_transform(prepared_frame)
    features = np.asarray(transformed, dtype=np.float32)

    labels = cleaned["label"].to_numpy(
        dtype="float64") if "label" in cleaned.columns else None
    meta = cleaned[["_source_file", "_source_folder", "_row_order"]].copy()
    return features, labels, meta, preprocessor




## Utility Functions: Sequence Creation

These functions convert row-level features into sliding-window temporal sequences and propagate labels to each window. They are needed for sequence-aware anomaly models and temporal context.

In CI/CD terms, each sequence can be interpreted as an ordered progression of pipeline stages over time (for example, checkout -> build -> test -> package -> deploy). This temporal framing preserves dependencies between stages and enables the models to learn normal end-to-end pipeline behavior rather than isolated events.


In [ ]:
def creating_sequences(features: np.ndarray, labels: Optional[np.ndarray], meta: pd.DataFrame, window_size: int = WINDOW_SIZE) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    sequence_list: List[np.ndarray] = []
    label_list: List[float] = []

    grouped_indices = meta.groupby("_source_file", sort=False).groups
    for _, indices in grouped_indices.items():
        ordered_indices = np.array(list(indices))
        ordered_indices = ordered_indices[np.argsort(
            meta.iloc[ordered_indices]["_row_order"].to_numpy())]
        if len(ordered_indices) < window_size:
            continue

        for start in range(0, len(ordered_indices) - window_size + 1):
            window_indices = ordered_indices[start: start + window_size]
            sequence_list.append(features[window_indices])
            if labels is not None:
                window_labels = labels[window_indices]
                known_labels = window_labels[~np.isnan(window_labels)]
                if len(known_labels) == 0:
                    label_list.append(np.nan)
                else:
                    label_list.append(float(np.max(known_labels)))

    if not sequence_list:
        raise RuntimeError(
            "Could not create any sequences. Increase data volume or reduce the window size.")

    sequences = np.asarray(sequence_list, dtype=np.float32)
    sequence_labels = np.asarray(
        label_list, dtype=np.float64) if labels is not None else None
    print(
        f"[SEQ] Created {len(sequences)} sequences with shape {sequences.shape[1:]}")
    return sequences, sequence_labels




## Utility Functions: Sampling and Splitting

These functions control runtime via sampling and partition data into train/test sets with stratification when possible. They are needed for efficient and reproducible training/evaluation.


In [ ]:
def checking_sample_if_needed(array: np.ndarray, max_samples: int, label: str) -> np.ndarray:
    if len(array) <= max_samples:
        return array
    rng = np.random.default_rng(SEED)
    indices = rng.choice(len(array), size=max_samples, replace=False)
    print(
        f"[SAMPLE] Reduced {label} from {len(array)} to {max_samples} samples for runtime control")
    return array[indices]


def spliting_sequences(sequences: np.ndarray, labels: Optional[np.ndarray]) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
    if labels is None or np.all(np.isnan(labels)):
        train_sequences, test_sequences = train_test_split(
            sequences, test_size=TEST_SIZE, random_state=SEED, shuffle=True)
        return train_sequences, test_sequences, None, None

    valid_mask = ~np.isnan(labels)
    labeled_sequences = sequences[valid_mask]
    labeled_labels = labels[valid_mask].astype(int)

    unique_labels, counts = np.unique(labeled_labels, return_counts=True)
    stratify = labeled_labels if len(
        unique_labels) > 1 and counts.min() >= 2 else None
    train_sequences, test_sequences, train_labels, test_labels = train_test_split(
        labeled_sequences,
        labeled_labels,
        test_size=TEST_SIZE,
        random_state=SEED,
        shuffle=True,
        stratify=stratify,
    )
    return train_sequences, test_sequences, train_labels, test_labels


def checking_flatten_sequences(sequences: np.ndarray) -> np.ndarray:
    return sequences.reshape(sequences.shape[0], -1)




## Model Definitions: Isolation Forest

Isolation Forest is an unsupervised baseline that isolates rare patterns quickly in high-dimensional spaces. It is used as a strong classical anomaly detector for comparison.


In [ ]:
def predicting_isolation_forest(model: IsolationForest, test_sequences: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    test_flat = checking_flatten_sequences(test_sequences)
    scores = -model.decision_function(test_flat)
    predictions = (model.predict(test_flat) == -1).astype(int)
    return predictions, scores




## Model Definitions: Autoencoder

The dense autoencoder learns to reconstruct normal behavior; high reconstruction error indicates anomalies. It is used to capture nonlinear structure in flattened sequence features.


In [ ]:
def building_autoencoder(input_dim: int) -> keras.Model:
    inputs = keras.Input(shape=(input_dim,), name="ae_input")
    x = keras.layers.Dense(256, activation="relu")(inputs)
    x = keras.layers.Dropout(0.2)(x)
    x = keras.layers.Dense(128, activation="relu")(x)
    x = keras.layers.Dropout(0.2)(x)
    x = keras.layers.Dense(64, activation="relu", name="ae_bottleneck")(x)
    x = keras.layers.Dense(128, activation="relu")(x)
    x = keras.layers.Dropout(0.2)(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    outputs = keras.layers.Dense(
        input_dim, activation="linear", name="ae_output")(x)
    model = keras.Model(inputs, outputs, name="dense_autoencoder")
    model.compile(optimizer=keras.optimizers.Adam(
        learning_rate=1e-3), loss="mse")
    return model




## Model Definitions: VAE-LSTM

The VAE-LSTM encodes temporal windows into a latent distribution and reconstructs sequences via LSTM decoder layers. It is used to model sequential dynamics and uncertainty for anomaly scoring.

For CI/CD security analytics, this architecture is well-suited to modeling stage-to-stage execution behavior, where deviations in timing, transitions, or reconstruction patterns can indicate abnormal builds, unstable release flows, or suspicious pipeline activity.


## Application in CI/CD Pipelines

This anomaly detection pipeline can be integrated with CI/CD orchestration platforms such as Jenkins and GitHub Actions by ingesting execution logs and stage telemetry from each run. In practical use, it can help flag abnormal builds, failed or high-risk deployments, and suspicious pipeline activity patterns that deviate from established operational baselines.


In [ ]:
class Sampling(keras.layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = keras.random.normal(shape=keras.ops.shape(z_mean))
        return z_mean + keras.ops.exp(0.5 * z_log_var) * epsilon


def building_vae_lstm(window_size: int, feature_dim: int, latent_dim: int = 32) -> Tuple[keras.Model, keras.Model, keras.Model]:
    encoder_inputs = keras.Input(
        shape=(window_size, feature_dim), name="vae_encoder_input")
    x = keras.layers.LSTM(128, return_sequences=True)(encoder_inputs)
    x = keras.layers.Dropout(0.2)(x)
    x = keras.layers.LSTM(64, return_sequences=False)(x)
    x = keras.layers.Dropout(0.2)(x)
    z_mean = keras.layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = keras.layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling(name="z")([z_mean, z_log_var])
    encoder = keras.Model(
        encoder_inputs, [z_mean, z_log_var, z], name="vae_encoder")

    latent_inputs = keras.Input(shape=(latent_dim,), name="vae_decoder_input")
    x = keras.layers.Dense(64, activation="relu")(latent_inputs)
    x = keras.layers.RepeatVector(window_size)(x)
    x = keras.layers.LSTM(64, return_sequences=True)(x)
    x = keras.layers.Dropout(0.2)(x)
    x = keras.layers.LSTM(128, return_sequences=True)(x)
    decoder_outputs = keras.layers.TimeDistributed(
        keras.layers.Dense(feature_dim, activation="linear"))(x)
    decoder = keras.Model(latent_inputs, decoder_outputs, name="vae_decoder")

    vae_inputs = keras.Input(
        shape=(window_size, feature_dim), name="vae_input")
    z_mean, z_log_var, z = encoder(vae_inputs)
    reconstruction = decoder(z)
    vae = keras.Model(vae_inputs, reconstruction, name="vae_lstm")
    vae.compile(optimizer=keras.optimizers.Adam(
        learning_rate=1e-3), loss="mse")
    return vae, encoder, decoder




In [ ]:
def predicting_autoencoder(model: keras.Model, test_sequences: np.ndarray, threshold: float) -> Tuple[np.ndarray, np.ndarray]:
    test_flat = checking_flatten_sequences(test_sequences)
    reconstruction = model.predict(test_flat, verbose=0)
    scores = np.mean(np.square(reconstruction - test_flat), axis=1)
    predictions = (scores > threshold).astype(int)
    return predictions, scores


def predicting_vae_lstm(vae: keras.Model, test_sequences: np.ndarray, threshold: float) -> Tuple[np.ndarray, np.ndarray]:
    reconstructed = vae.predict(test_sequences, verbose=0)
    scores = np.mean(np.square(reconstructed - test_sequences), axis=(1, 2))
    predictions = (scores > threshold).astype(int)
    return predictions, scores




## Training Functions

These functions train each model with the same runtime controls (normal-only training when labels exist, capped sampling, and early stopping for neural models).


In [ ]:
def training_isolation_forest(train_sequences: np.ndarray, train_labels: Optional[np.ndarray]) -> IsolationForest:
    train_flat = checking_flatten_sequences(train_sequences)
    if train_labels is not None and np.any(train_labels == 0):
        train_flat = train_flat[train_labels == 0]
        print(f"[IF] Training on {len(train_flat)} normal sequences")
    else:
        print(
            f"[IF] Training on all {len(train_flat)} sequences (unsupervised fallback)")

    train_flat = checking_sample_if_needed(
        train_flat, MAX_TRAIN_SAMPLES, "Isolation Forest training data")
    model = IsolationForest(
        n_estimators=200, contamination="auto", random_state=SEED, n_jobs=-1)
    model.fit(train_flat)
    return model




In [ ]:
def train_autoencoder(train_sequences: np.ndarray, train_labels: Optional[np.ndarray]) -> Tuple[keras.Model, keras.callbacks.History, float]:
    train_flat = checking_flatten_sequences(train_sequences)
    if train_labels is not None and np.any(train_labels == 0):
        train_flat = train_flat[train_labels == 0]
        print(f"[AE] Training on {len(train_flat)} normal sequences")
    else:
        print(
            f"[AE] Training on all {len(train_flat)} sequences (unsupervised fallback)")

    train_flat = checking_sample_if_needed(
        train_flat, MAX_TRAIN_SAMPLES, "Autoencoder training data")
    model = building_autoencoder(train_flat.shape[1])
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True),
    ]
    history = model.fit(
        train_flat,
        train_flat,
        epochs=AUTOENCODER_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1 if len(train_flat) >= 20 else 0.0,
        verbose=1,
        callbacks=callbacks,
    )
    train_reconstruction = np.mean(
        np.square(model.predict(train_flat, verbose=0) - train_flat), axis=1)
    threshold = float(np.percentile(train_reconstruction, 95))
    return model, history, threshold




In [ ]:
def training_vae_lstm(train_sequences: np.ndarray, train_labels: Optional[np.ndarray]) -> Tuple[keras.Model, keras.Model, keras.callbacks.History, float]:
    training_data = train_sequences
    if train_labels is not None and np.any(train_labels == 0):
        training_data = train_sequences[train_labels == 0]
        print(f"[VAE] Training on {len(training_data)} normal sequences")
    else:
        print(
            f"[VAE] Training on all {len(training_data)} sequences (unsupervised fallback)")

    training_data = checking_sample_if_needed(
        training_data, MAX_TRAIN_SAMPLES, "VAE training data")
    vae, encoder, decoder = building_vae_lstm(
        training_data.shape[1], training_data.shape[2])
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True),
    ]
    history = vae.fit(
        training_data,
        training_data,
        epochs=VAE_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1 if len(training_data) >= 20 else 0.0,
        verbose=1,
        callbacks=callbacks,
    )
    train_reconstruction = vae.predict(training_data, verbose=0)
    train_scores = np.mean(
        np.square(train_reconstruction - training_data), axis=(1, 2))
    threshold = float(np.percentile(train_scores, 95))
    return vae, encoder, history, threshold




## Evaluation and Visualization

This section computes classification metrics (Accuracy, Precision, Recall, F1, ROC-AUC), prints summaries, and saves confusion matrices, ROC curves, loss curves, and anomaly score distributions.

Because publicly available, standardized CI/CD security datasets are still limited, this study uses heterogeneous anomaly-oriented datasets to evaluate model behavior under varied abnormal conditions. Although these datasets are not exclusively pipeline-native, the learned anomaly detection mechanisms remain applicable to CI/CD environments where sequential logs and execution traces exhibit similar behavioral structure.


In [ ]:
def safe_metrics_data(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray) -> dict:
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        if len(np.unique(y_true)) > 1:
            metrics["roc_auc"] = roc_auc_score(y_true, y_score)
        else:
            metrics["roc_auc"] = float("nan")
    except Exception:
        metrics["roc_auc"] = float("nan")
    metrics["confusion_matrix"] = confusion_matrix(y_true, y_pred)
    return metrics


def print_metrics_data(model_name: str, metrics: dict) -> None:
    print(f"\n=== {model_name} Evaluation ===")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1-score : {metrics['f1']:.4f}")
    roc_auc = metrics.get("roc_auc", float("nan"))
    if math.isnan(roc_auc):
        print("ROC-AUC  : unavailable")
    else:
        print(f"ROC-AUC  : {roc_auc:.4f}")
    print("Confusion Matrix:")
    print(metrics["confusion_matrix"])


def save_confusion_matrix_plot_data(cm: np.ndarray, title: str, file_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    tick_labels = ["Normal", "Anomaly"]
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(tick_labels)
    ax.set_yticklabels(tick_labels)

    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            ax.text(col, row, str(cm[row, col]),
                    ha="center", va="center", color="black")

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(file_path, dpi=200)
    plt.close(fig)


def save_roc_curve_plot_data(y_true: np.ndarray, y_score: np.ndarray, title: str, file_path: Path) -> None:
    if len(np.unique(y_true)) < 2:
        return
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc_value = roc_auc_score(y_true, y_score)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f"ROC-AUC = {auc_value:.4f}", color="#1f77b4")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set_title(title)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(file_path, dpi=200)
    plt.close(fig)


def save_loss_curve_data(history, title: str, file_path: Path, include_components: bool = False) -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    loss = history.history.get("loss", [])
    val_loss = history.history.get("val_loss", [])
    ax.plot(loss, label="train_loss", color="#1f77b4")
    if val_loss:
        ax.plot(val_loss, label="val_loss", color="#ff7f0e")
    if include_components:
        reconstruction_loss = history.history.get("reconstruction_loss", [])
        kl_loss = history.history.get("kl_loss", [])
        if reconstruction_loss:
            ax.plot(reconstruction_loss,
                    label="reconstruction_loss", color="#2ca02c")
        if kl_loss:
            ax.plot(kl_loss, label="kl_loss", color="#d62728")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    fig.tight_layout()
    fig.savefig(file_path, dpi=200)
    plt.close(fig)


def save_score_distribution(y_true: Optional[np.ndarray], scores: np.ndarray, title: str, file_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    if y_true is not None and len(np.unique(y_true)) > 1:
        normal_scores = scores[y_true == 0]
        anomaly_scores = scores[y_true == 1]
        ax.hist(normal_scores, bins=40, alpha=0.65,
                label="Normal", density=True, color="#2ca02c")
        ax.hist(anomaly_scores, bins=40, alpha=0.65,
                label="Anomaly", density=True, color="#d62728")
        ax.legend()
    else:
        ax.hist(scores, bins=40, alpha=0.8, color="#1f77b4", density=True)
    ax.set_title(title)
    ax.set_xlabel("Anomaly Score")
    ax.set_ylabel("Density")
    fig.tight_layout()
    fig.savefig(file_path, dpi=200)
    plt.close(fig)


def evaluate_and_save_data(model_name: str, y_true: Optional[np.ndarray], y_pred: np.ndarray, y_score: np.ndarray, output_dir: Path) -> Optional[dict]:
    artifact_paths = artifact_paths_for_model(model_name, output_dir)
    if y_true is None or len(y_true) == 0 or len(np.unique(y_true[~np.isnan(y_true)])) < 2:
        print(
            f"[EVAL] {model_name}: insufficient labels for full metric evaluation.")
        save_score_distribution(None, y_score, f"{model_name} Anomaly Score Distribution",
                                artifact_paths["anomaly_scores"])
        print(
            f"[SAVE] Saved anomaly scores plot: {artifact_paths['anomaly_scores']}")
        return None

    valid_mask = ~np.isnan(y_true)
    y_true = y_true[valid_mask].astype(int)
    y_pred = y_pred[valid_mask].astype(int)
    y_score = y_score[valid_mask].astype(float)

    metrics = safe_metrics_data(y_true, y_pred, y_score)
    print_metrics_data(model_name, metrics)
    save_confusion_matrix_plot_data(metrics["confusion_matrix"], f"{model_name} Confusion Matrix",
                                    artifact_paths["confusion_matrix"])
    print(
        f"[SAVE] Saved confusion matrix plot: {artifact_paths['confusion_matrix']}")
    save_roc_curve_plot_data(y_true, y_score, f"{model_name} ROC Curve",
                             artifact_paths["roc_curve"])
    print(f"[SAVE] Saved ROC curve plot: {artifact_paths['roc_curve']}")
    save_score_distribution(y_true, y_score, f"{model_name} Anomaly Score Distribution",
                            artifact_paths["anomaly_scores"])
    print(
        f"[SAVE] Saved anomaly scores plot: {artifact_paths['anomaly_scores']}")
    return {
        "Model": model_name,
        "Accuracy": metrics["accuracy"],
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1": metrics["f1"],
        "ROC-AUC": metrics["roc_auc"],
    }


def save_model_comparison(rows: List[dict], output_dir: Path) -> None:
    if not rows:
        return
    comparison = pd.DataFrame(
        rows, columns=["Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"])
    comparison_path = output_dir / "model_comparison.csv"
    comparison.to_csv(comparison_path, index=False)
    print(f"[SAVE] Saved model comparison CSV: {comparison_path}")
    print("\n[EVAL] Model Comparison")
    print(comparison.to_string(index=False,
          float_format=lambda value: f"{value:.4f}"))




## Main Pipeline Execution

This final execution cell runs the complete pipeline end-to-end.


In [ ]:
def run_pipeline() -> None:
    setting_global_seed()
    output_dir = ensure_output_directory()
    comparison_rows: List[dict] = []
    print("[LOAD] Discovering CSV files...")
    csv_files = discovering_csv_files(DATASET_DIRS)

    if not csv_files:
        print("[CI/CD MODE] No datasets found. Skipping full pipeline execution.")
        print("[CI/CD MODE] Pipeline executed successfully (simulation mode).")
        import os
        os.makedirs("outputs", exist_ok=True)
        with open("outputs/dummy_output.txt", "w") as f:
            f.write("CI/CD run successful without datasets.")
        return

    print(f"[LOAD] Found {len(csv_files)} CSV files")

    print("[LOAD] Loading datasets...")
    combined = loading_all_datasets(csv_files)
    print("[PREPROCESS] Preprocessing and encoding features...")
    features, labels, meta, _ = preprocess_dataframe(combined)

    print("[SEQ] Creating sliding-window sequences...")
    sequences, sequence_labels = creating_sequences(
        features, labels, meta, window_size=WINDOW_SIZE)

    train_sequences, test_sequences, train_labels, test_labels = spliting_sequences(
        sequences, sequence_labels)
    print(
        f"[SPLIT] Train sequences: {len(train_sequences)}, Test sequences: {len(test_sequences)}")

    print("[MODEL] Training Isolation Forest baseline...")
    isolation_forest = training_isolation_forest(train_sequences, train_labels)
    if_predictions, if_scores = predicting_isolation_forest(
        isolation_forest, test_sequences)
    metrics_row = evaluate_and_save_data("Isolation Forest", test_labels,
                                        if_predictions, if_scores, output_dir)
    if metrics_row:
        comparison_rows.append(metrics_row)

    print("[MODEL] Training Autoencoder...")
    autoencoder, auto_history, ae_threshold = train_autoencoder(
        train_sequences, train_labels)
    ae_predictions, ae_scores = predicting_autoencoder(
        autoencoder, test_sequences, ae_threshold)
    metrics_row = evaluate_and_save_data("Autoencoder", test_labels,
                                       ae_predictions, ae_scores, output_dir)
    if metrics_row:
        comparison_rows.append(metrics_row)
    ae_artifacts = artifact_paths_for_model("Autoencoder", output_dir)
    save_loss_curve_data(
        auto_history,
        "Autoencoder Loss Curve",
        ae_artifacts["loss_curve"],
    )
    print(f"[SAVE] Saved loss curve plot: {ae_artifacts['loss_curve']}")

    print("[MODEL] Training VAE-LSTM...")
    vae, _, vae_history, vae_threshold = training_vae_lstm(
        train_sequences, train_labels)
    vae_predictions, vae_scores = predicting_vae_lstm(
        vae, test_sequences, vae_threshold)
    metrics_row = evaluate_and_save_data("VAE-LSTM", test_labels,
                                    vae_predictions, vae_scores, output_dir)
    if metrics_row:
        comparison_rows.append(metrics_row)
    vae_artifacts = artifact_paths_for_model("VAE-LSTM", output_dir)
    save_loss_curve_data(
        vae_history,
        "VAE-LSTM Loss Curve",
        vae_artifacts["loss_curve"],
    )
    print(f"[SAVE] Saved loss curve plot: {vae_artifacts['loss_curve']}")

    save_model_comparison(comparison_rows, output_dir)

    print("[DONE] Pipeline completed successfully.")
    print(f"[DONE] Plots and artifacts saved to: {output_dir}")



run_pipeline()


## Output Explanation

After execution, artifacts are saved in the outputs/ folder:

- Plots: confusion matrices, ROC curves, loss curves, anomaly score distributions
- CSV: model_comparison.csv

## Conclusion

The comparative results from Isolation Forest, Autoencoder, and VAE-LSTM provide a practical foundation for anomaly-aware monitoring in CI/CD workflows. By learning normal execution behavior and highlighting deviations, the pipeline supports earlier detection of abnormal builds, unstable deployment paths, and potentially suspicious automation activity. This strengthens DevSecOps posture by adding data-driven behavioral monitoring to existing build and release controls.
